# Experiment: Router-train uniform MoE-DGA expert predictions

Router-train upload in Colab:

- `router_train_colab_content_root_package.zip`

For router-training predictions, upload the package that contains `router_train_domains.csv` and set in the configuration cell:

- `RUN_NAME = "router_train"`
- `DATASET_NAME = "router_train_domains.csv"`

The package must be uploaded to `/content` and contain a `New_Paper_MOE/` folder with:

- `New_Paper_MOE/data/<dataset>.csv`
- `New_Paper_MOE/data/family_taxonomy.csv`
- `New_Paper_MOE/notebooks/router_train_colab_uniform_predictions.ipynb`

Objective:

- Run `Logit`, `ModernBERT`, and `DomURLs-BERT` on the same Colab hardware.
- Export one prediction table with identical domains, labels, scores, and latency fields.
- Produce the input needed for single-model, ensemble, oracle, cascade, random-router, and cost-quality analyses.

Success criteria:

- Same domain set for every expert.
- Valid `p_dga`, `y_pred`, and `latency_ms` columns.
- Results separable by `WL`, `NW`, and benign traffic.


## 1. Runtime Notes

Before running, set Colab to the target hardware. Keep all experts in the same session when possible. If memory forces separate sessions, rerun the same metadata cell and keep the hardware identical.

Upload only `router_train_colab_content_root_package.zip`; do not upload the full project folder.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import platform
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

runtime_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor(),
}
runtime_info


In [ ]:
# Optional Colab GPU metadata. Safe if nvidia-smi is unavailable.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true


## 2. Upload Package Check

After uploading `router_train_colab_content_root_package.zip` to Colab, run the next cell. It extracts the package into `/content/New_Paper_MOE` and verifies that the two required CSV files are present.


In [ ]:
PACKAGE_ZIP = Path("/content/router_train_colab_content_root_package.zip")
PROJECT_ROOT = Path("/content/New_Paper_MOE")

if PACKAGE_ZIP.exists():
    !unzip -o /content/router_train_colab_content_root_package.zip -d /content
elif not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Upload router_train_colab_content_root_package.zip to Colab, then rerun this cell."
    )

required_files = [
    PROJECT_ROOT / "data" / "family_taxonomy.csv",
    PROJECT_ROOT / "data" / "router_train_domains.csv",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required router-train input files: " + ", ".join(missing_files)
    )

for path in required_files:
    print(path, path.stat().st_size, "bytes")


In [ ]:
# Run once in Colab before loading DomURLs-BERT.
# DomURLs-BERT uses PEFT LoRA but does not need torchao quantization.
# Some Colab images ship torchao 0.10.0, which makes recent peft fail during import.
%pip uninstall -y torchao
%pip install -q -U transformers peft huggingface_hub joblib scikit-learn scipy accelerate


## 3. Configuration

Upload or mount the project assets so these paths exist. Do not write into the original ARGENCON or IEEE LATAM folders; export only the prediction CSV.


In [ ]:
# The upload-check cell defines this path for Colab.
PROJECT_ROOT = Path("/content/New_Paper_MOE")

# Router training run. Do not change these values for this notebook.
RUN_NAME = "router_train"
DATASET_NAME = "router_train_domains.csv"

TAXONOMY_CSV = PROJECT_ROOT / "data" / "family_taxonomy.csv"
DATASET_CSV = PROJECT_ROOT / "data" / DATASET_NAME
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "predictions"
OUTPUT_CSV = OUTPUT_DIR / f"{RUN_NAME}_uniform_predictions.csv"
RUN_METADATA_JSON = OUTPUT_DIR / f"{RUN_NAME}_runtime_metadata.json"

for required_path in [TAXONOMY_CSV, DATASET_CSV]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing {required_path}. Upload the Colab package and run the upload-check cell."
        )

EXPERTS = ["Logit", "ModernBERT", "DomURLs-BERT"]
HF_REPOS = {
    "Logit": "Reynier/dga-logit",
    "ModernBERT": "Reynier/modernbert-dga-detector",
    "DomURLs-BERT": "Reynier/dga-domurlsbert",
}
BATCH_SIZE = {
    "Logit": 4096,
    "ModernBERT": 128,
    "DomURLs-BERT": 128,
}
THRESHOLD = 0.5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
{
    "run_name": RUN_NAME,
    "taxonomy": str(TAXONOMY_CSV),
    "dataset": str(DATASET_CSV),
    "output": str(OUTPUT_CSV),
    "experts": EXPERTS,
}


## 4. Load Taxonomy And Dataset

`router_train_domains.csv` must contain at least `domain`, `family`, `dga_type`, `split`, and `y_true`. This run uses seen-family `Families_Test` domains plus benign domains for router training.


In [ ]:
taxonomy = pd.read_csv(TAXONOMY_CSV)
if "dga_type" not in taxonomy.columns and "type" in taxonomy.columns:
    taxonomy = taxonomy.rename(columns={"type": "dga_type"})
if "dga_type" not in taxonomy.columns:
    raise ValueError(f"Taxonomy must include dga_type or type. Columns: {list(taxonomy.columns)}")
display(taxonomy.groupby(["split", "dga_type"]).size().rename("families").reset_index())

heldout_families = taxonomy.loc[taxonomy["split"].eq("heldout"), "family"].tolist()
heldout_families[:10], len(heldout_families)


In [ ]:
domains = pd.read_csv(DATASET_CSV)
required = {"domain", "family", "dga_type", "split", "y_true"}
missing = required - set(domains.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

domains["domain"] = domains["domain"].astype(str).str.strip().str.lower()
domains = domains.drop_duplicates(subset=["domain", "family", "split"]).reset_index(drop=True)

display(domains.groupby(["split", "dga_type"]).size().rename("domains").reset_index())
domains.head()


## 5. Expert Adapters

Replace each loader with the actual project code/model path. Every prediction function must accept a list of domains and return a NumPy array of `p_dga` scores in the same order.


In [ ]:
@dataclass
class ExpertAdapter:
    name: str
    predict_proba: Callable[[list[str]], np.ndarray]


def load_logit() -> ExpertAdapter:
    from huggingface_hub import hf_hub_download

    repo_id = HF_REPOS["Logit"]
    artifacts_path = hf_hub_download(repo_id, "artifacts.joblib")
    model_py = hf_hub_download(repo_id, "model.py")
    spec = importlib.util.spec_from_file_location("dga_logit_model", model_py)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    artifacts = module.load_model(artifacts_path)

    def predict_proba(batch: list[str]) -> np.ndarray:
        results = module.predict(artifacts, batch)
        return np.asarray([float(item["score"]) for item in results], dtype=float)

    return ExpertAdapter("Logit", predict_proba)


def load_modernbert() -> ExpertAdapter:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    repo_id = HF_REPOS["ModernBERT"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    model = AutoModelForSequenceClassification.from_pretrained(repo_id).to(device).eval()

    def predict_proba(batch: list[str]) -> np.ndarray:
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
            scores = torch.softmax(logits, dim=1)[:, 1]
        return scores.detach().cpu().numpy().astype(float)

    return ExpertAdapter("ModernBERT", predict_proba)


def load_domurls_bert() -> ExpertAdapter:
    import torch
    from peft import PeftModel
    from transformers import BertForSequenceClassification, BertTokenizer

    repo_id = HF_REPOS["DomURLs-BERT"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = BertTokenizer.from_pretrained(repo_id)
    base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    model = PeftModel.from_pretrained(base_model, repo_id).to(device).eval()

    def predict_proba(batch: list[str]) -> np.ndarray:
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
            scores = torch.softmax(logits, dim=1)[:, 1]
        return scores.detach().cpu().numpy().astype(float)

    return ExpertAdapter("DomURLs-BERT", predict_proba)


LOADERS = {
    "Logit": load_logit,
    "ModernBERT": load_modernbert,
    "DomURLs-BERT": load_domurls_bert,
}


## 6. Inference Helpers

The timing is measured around batches. The same per-domain latency is assigned to every row in that batch.


In [ ]:
def batched(items: list[str], batch_size: int):
    for start in range(0, len(items), batch_size):
        yield start, items[start:start + batch_size]


def warmup(adapter: ExpertAdapter, examples: list[str], batch_size: int, rounds: int = 2) -> None:
    warm_batch = examples[: min(len(examples), batch_size)]
    for _ in range(rounds):
        _ = adapter.predict_proba(warm_batch)


def run_expert(adapter: ExpertAdapter, frame: pd.DataFrame, batch_size: int) -> pd.DataFrame:
    domains_list = frame["domain"].tolist()
    scores = np.empty(len(domains_list), dtype=float)
    latency_ms = np.empty(len(domains_list), dtype=float)

    warmup(adapter, domains_list, batch_size)

    for start, batch in batched(domains_list, batch_size):
        t0 = time.perf_counter()
        batch_scores = np.asarray(adapter.predict_proba(batch), dtype=float)
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        if len(batch_scores) != len(batch):
            raise ValueError(f"{adapter.name} returned {len(batch_scores)} scores for {len(batch)} domains")
        end = start + len(batch)
        scores[start:end] = batch_scores
        latency_ms[start:end] = elapsed_ms / max(len(batch), 1)

    out = frame.copy()
    out["model"] = adapter.name
    out["p_dga"] = scores
    out["y_pred"] = (out["p_dga"] >= THRESHOLD).astype(int)
    out["latency_ms"] = latency_ms
    return out


## 7. Run Experts

Run one expert at a time if GPU memory is tight. Append all outputs and save one final CSV.


In [ ]:
all_predictions = []

for expert_name in EXPERTS:
    print(f"Loading {expert_name}")
    adapter = LOADERS[expert_name]()
    print(f"Running {expert_name}")
    pred = run_expert(adapter, domains, BATCH_SIZE[expert_name])
    all_predictions.append(pred)
    display(pred.groupby(["dga_type"])["latency_ms"].describe()[["mean", "50%", "max"]])

predictions = pd.concat(all_predictions, ignore_index=True)
predictions.head()


## 8. Sanity Checks

These checks must pass before using the file for Gate 1 analysis.


In [ ]:
expected_rows = len(domains) * len(EXPERTS)
if len(predictions) != expected_rows:
    raise ValueError(f"Expected {expected_rows} rows, got {len(predictions)}")

if predictions.duplicated(subset=["model", "domain", "family", "split"]).any():
    raise ValueError("Duplicated model/domain/family/split rows found")

if predictions["p_dga"].isna().any():
    raise ValueError("Null p_dga values found")

if not predictions["p_dga"].between(0, 1).all():
    raise ValueError("p_dga must be in [0, 1]")

if (predictions["latency_ms"] < 0).any():
    raise ValueError("Negative latency found")

summary = predictions.groupby(["model", "dga_type"]).agg(
    rows=("domain", "size"),
    pred_positive_rate=("y_pred", "mean"),
    latency_mean_ms=("latency_ms", "mean"),
    latency_p50_ms=("latency_ms", "median"),
    latency_p95_ms=("latency_ms", lambda s: float(np.percentile(s, 95))),
    latency_p99_ms=("latency_ms", lambda s: float(np.percentile(s, 99))),
).reset_index()
display(summary)


## 9. Export

Save the prediction table and metadata. Bring these files back into the repository under `experiments/predictions/`.


In [ ]:
columns = [
    "domain",
    "family",
    "dga_type",
    "split",
    "y_true",
    "model",
    "y_pred",
    "p_dga",
    "latency_ms",
]
predictions[columns].to_csv(OUTPUT_CSV, index=False)

metadata = {
    "seed": SEED,
    "experts": EXPERTS,
    "batch_size": BATCH_SIZE,
    "threshold": THRESHOLD,
    "runtime": runtime_info,
    "num_input_domains": int(len(domains)),
    "num_output_rows": int(len(predictions)),
    "output_csv": str(OUTPUT_CSV),
}
RUN_METADATA_JSON.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

str(OUTPUT_CSV), str(RUN_METADATA_JSON)


## 10. Next Steps

After exporting from Colab:

- download `gate1_uniform_predictions.csv` and `gate1_runtime_metadata.json`;
- place both files in `experiments/predictions/` in this repository;
- run `python3 scripts/analyze_gate1_predictions.py` locally;
- update `analisis/bitacora_moe_dga.md` with the Colab runtime and output files.
